# Cash-Flow Forecasting — Full Pipeline

Per-user Prophet models that predict daily inflow, detect income dips, and size micro-loan offers.

## Steps
1. **Load** transactions (22.7M rows, 10k users) and inspect the schema
2. **Aggregate** raw transactions into a daily inflow series per user
3. **Archetypes** — build cold-start profiles from population data
4. **Fit** Prophet on one user — smoke test before scaling to 10k fits
5. **Backtest** — 500-user holdout; MAE, MASE, 80% coverage
6. **Dip detection** — identify sustained income drops in the forecast horizon
7. **Loan sizing** — size a micro-loan offer from the predicted income gap
8. **End-to-end** — full `CashFlowPredictor` path: cache → lazy fit → dip warning

---
> **Decisions locked in**
> - One Prophet model per user, fit on log1p-transformed naira
> - 21-day minimum history; below that → archetype day-of-week baseline
> - Custom seasonalities: weekly (28d+), monthly (60d+), NG public holidays + EOM paydays
> - 80% prediction interval; dip = `yhat_lower < 50% of P25 baseline` for 3+ consecutive days
> - Loan = gap × 1.20, rounded to ₦5k, floor ₦10k, capped by KudiScore

In [ ]:
import sys, time, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 60)

DATA_DIR   = Path('../data')
MODELS_DIR = Path('../models')

print('Ready.')

## Step 1 — Load Data

22.7M transactions for 10,000 synthetic Nigerian trader profiles across 6 archetypes.
Column contract: `user_id`, `occurred_at`, `amount_kobo`, `type` (`inflow` | `outflow`).

Load the users table alongside transactions — you'll need the archetype labels for the
cold-start profiles and for filtering by history length.

In [ ]:
t0 = time.time()
tx = pd.read_parquet(DATA_DIR / 'synth_transactions.parquet')
tx['occurred_at'] = pd.to_datetime(tx['occurred_at'])

users = pd.read_parquet(DATA_DIR / 'synth_users.parquet')
users['onboarding_date'] = pd.to_datetime(users['onboarding_date'])

print(f'Loaded in {time.time()-t0:.1f}s')
print(f'Transactions: {len(tx):,}')
print(f'Users:        {tx.user_id.nunique():,}')
print(f'Date range:   {tx.occurred_at.min().date()} → {tx.occurred_at.max().date()}')
print()
print('Transaction schema:')
print(tx.dtypes.to_string())
print()
print('Type breakdown:')
print(tx['type'].value_counts().to_string())
print()
tx.head(3)

In [ ]:
# How many days of inflow history does each user have?
inflow_days = (
    tx[tx['type'] == 'inflow']
    .groupby('user_id')['occurred_at']
    .agg(lambda x: (x.max() - x.min()).days + 1)
    .rename('history_days')
)

bins = [0, 21, 28, 60, 90, 120, 180, 999]
labels_hist = ['<21d (archetype only)', '21-28d', '28-60d',
               '60-90d', '90-120d', '120-180d', '>180d']
hist_cut = pd.cut(inflow_days, bins=bins, labels=labels_hist)

print('History length distribution (determines which Prophet features activate):')
print(hist_cut.value_counts().sort_index().to_string())
print()
print(f'Users with ≥21d (Prophet-eligible):  {(inflow_days >= 21).sum():,}')
print(f'Users with ≥28d (weekly seasonality): {(inflow_days >= 28).sum():,}')
print(f'Users with ≥60d (monthly seasonality): {(inflow_days >= 60).sum():,}')
print(f'Users with ≥90d (backtest-eligible):  {(inflow_days >= 90).sum():,}')

## Step 2 — Daily Series Aggregation

`build_daily_series` turns raw inflow transactions into a zero-filled daily series.
Everything downstream — Prophet fit, archetypes, backtest — takes this as input,
so get the aggregation right before moving on.

**Output contract:**
- `ds` — one row per calendar day, no gaps
- `y` / `y_orig` — daily inflow in **naira** (amount_kobo ÷ 100)
- Days with no transactions → `y = 0` (zero-fill, not NaN)

**Smoke test checklist:**
- No gaps in `ds`
- `y` is in naira range (thousands to hundreds of thousands per day)
- Zero days are expected (weekends, market closures) — not a bug

In [ ]:
from training.cash_flow import build_daily_series

# Pick the highest-volume user for a meaningful smoke test
counts = tx[tx['type'] == 'inflow'].groupby('user_id').size().sort_values(ascending=False)
DEMO_USER = counts.index[0]
DEMO_ARCHETYPE = users.loc[users['user_id'] == DEMO_USER, 'archetype'].iloc[0]

print(f'Demo user:    {DEMO_USER}')
print(f'Archetype:    {DEMO_ARCHETYPE}')
print(f'Inflow txns:  {counts.iloc[0]:,}')
print()

t0 = time.time()
daily = build_daily_series(tx, DEMO_USER)
elapsed = time.time() - t0

print(f'Built in {elapsed:.2f}s')
print(f'Shape:        {daily.shape}')
print(f'Date range:   {daily.ds.min().date()} → {daily.ds.max().date()}')
print(f'Gaps in ds:   {(daily.ds.diff().dt.days.dropna() != 1).sum()} (expect 0)')
print(f'Zero days:    {(daily.y == 0).sum()}')
print(f'y min/max:    ₦{daily.y.min():,.0f} / ₦{daily.y.max():,.0f}')
print(f'y mean:       ₦{daily.y.mean():,.0f}')
print()
daily.tail(7)

In [ ]:
# Hard assertions — these should never fail on valid data
assert (daily.ds.diff().dt.days.dropna() == 1).all(), 'Gap found in daily series'
assert (daily.y >= 0).all(), 'Negative inflow found'
assert daily.y.max() < 10_000_000, 'y looks like it might still be in kobo (>₦10M/day)'
assert daily.y.mean() > 100, 'y too small — check kobo→naira division'
assert (daily.y == daily.y_orig).all(), 'y and y_orig should be identical before log transform'

print('✓ All daily-series assertions passed')

In [ ]:
# Run assertions across 20 random users to catch edge cases
sample_users = users['user_id'].sample(20, random_state=7).tolist()
failures = []

for uid in sample_users:
    d = build_daily_series(tx, uid)
    if d.empty:
        continue
    try:
        assert (d.ds.diff().dt.days.dropna() == 1).all()
        assert (d.y >= 0).all()
        assert d.y.max() < 10_000_000
    except AssertionError as e:
        failures.append((uid, str(e)))

print(f'Tested {len(sample_users)} users — {len(failures)} failures')
for uid, err in failures:
    print(f'  {uid}: {err}')
if not failures:
    print('✓ All pass')

## Step 3 — Archetype Profiles (Cold-Start Baseline)

If a user has fewer than 21 days of history, Prophet can't be fit.
The fallback is a **day-of-week profile** derived from all users sharing the same archetype —
mean ± std daily inflow per (archetype, day-of-week), and the 80% interval is mean ± 1.28 × std.

Build this once offline and save it to `data/archetype_profiles.json`.
The ML service loads it at startup; no refit ever happens at request time.

In [ ]:
from training.cash_flow import (
    build_archetype_profiles, save_archetype_profiles,
    load_archetype_profiles, archetype_forecast
)

profiles_path = DATA_DIR / 'archetype_profiles.json'

if profiles_path.exists():
    print('Loading pre-built archetype profiles...')
    profiles = load_archetype_profiles(profiles_path)
else:
    print('Building archetype profiles from full transaction history...')
    t0 = time.time()
    persona_map = dict(zip(users['user_id'], users['archetype']))
    profiles = build_archetype_profiles(tx, persona_map)
    save_archetype_profiles(profiles, profiles_path)
    print(f'Built in {time.time()-t0:.1f}s')

print(f'Profiles: {len(profiles)} archetypes')
print()

# Show mean daily inflow by archetype across the week
DOW = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
rows = []
for persona, prof in profiles.items():
    row = {'archetype': persona}
    for i, day in enumerate(DOW):
        row[day] = prof['by_dow_mean'].get(str(i), 0)
    row['avg'] = np.mean(list(prof['by_dow_mean'].values()))
    rows.append(row)

profile_df = pd.DataFrame(rows).set_index('archetype')
print('Mean daily inflow by archetype (₦):')
display(profile_df.applymap(lambda x: f'₦{x:,.0f}'))

In [ ]:
# Visualise archetype day-of-week pattern
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=False)

for ax, (persona, prof) in zip(axes.flat, profiles.items()):
    means = [prof['by_dow_mean'].get(str(i), 0) for i in range(7)]
    stds  = [prof['by_dow_std'].get(str(i), 0)  for i in range(7)]
    x = range(7)
    ax.bar(x, means, color='#2563EB', alpha=0.8)
    ax.errorbar(x, means, yerr=[s * 1.28 for s in stds],
                fmt='none', color='#1E3A8A', capsize=4, linewidth=1.5)
    ax.set_xticks(range(7))
    ax.set_xticklabels(DOW, fontsize=9)
    ax.set_title(persona.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.set_ylabel('Daily inflow (₦)', fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₦{x/1000:.0f}k'))
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Cold-Start Archetype Profiles — Mean ± 80% Interval by Day of Week',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_DIR / 'archetype_profiles.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/archetype_profiles.png')

In [ ]:
# Demo: cold-start forecast for a new user (archetype = market_food_vendor)
cold_start = archetype_forecast(
    persona='market_food_vendor',
    horizon_days=14,
    start_date=pd.Timestamp.utcnow().normalize().tz_localize(None),
    profiles=profiles,
)

print('Cold-start 14-day forecast for market_food_vendor:')
print(cold_start[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_string(index=False))
print()
print(f'Interval check (lower ≤ yhat ≤ upper): {((cold_start.yhat_lower <= cold_start.yhat) & (cold_start.yhat <= cold_start.yhat_upper)).all()}')
print(f'Any negatives in lower:                {(cold_start.yhat_lower < 0).any()}')

## Step 4 — Prophet Fit (Single User)

**This is the demo milestone.** Get a forecast rendering for one user before scaling to 10k.
Once this plot looks right, the hard part is done.

Key settings to be aware of:
- `changepoint_prior_scale=0.05` — conservative trend flexibility; avoids overfitting short series
- `interval_width=0.80` — 80% prediction interval
- Log1p transform applied before fit, inverted on output so `yhat` is back in naira
- NG public holidays + end-of-month payday bumps added as regressors

If `yhat_lower` comes back negative or in the millions, jump to the troubleshooting table
at the bottom of the notebook before continuing.

In [ ]:
from training.cash_flow import fit_user_forecast, build_eom_holidays

daily = build_daily_series(tx, DEMO_USER)
holidays = build_eom_holidays(
    daily['ds'].min().strftime('%Y-%m-%d'),
    (daily['ds'].max() + pd.Timedelta(days=60)).strftime('%Y-%m-%d'),
)

print(f'Fitting Prophet on {len(daily)} days of history...')
t0 = time.time()
model, forecast = fit_user_forecast(daily, horizon_days=30, custom_holidays=holidays)
print(f'Fit time: {time.time()-t0:.1f}s')
print()

future = forecast[forecast['ds'] > daily['ds'].max()].reset_index(drop=True)
print(f'Future rows: {len(future)}')
print(f'yhat range:  ₦{future.yhat.min():,.0f} – ₦{future.yhat.max():,.0f}')
print(f'Interval OK (lower ≤ yhat ≤ upper): {((future.yhat_lower <= future.yhat) & (future.yhat <= future.yhat_upper)).all()}')
print(f'Any negatives in lower: {(future.yhat_lower < 0).any()}')
print()
future[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].head(7)

In [ ]:
# Plot: actual history + 30-day forecast with 80% interval
fig, ax = plt.subplots(figsize=(14, 5))

# History (last 60 days for readability)
history_plot = daily.tail(60)
ax.plot(history_plot['ds'], history_plot['y_orig'],
        color='#374151', linewidth=1.2, alpha=0.8, label='Actual (last 60d)')

# Forecast
ax.plot(future['ds'], future['yhat'],
        color='#2563EB', linewidth=2, label='Forecast (yhat)')
ax.fill_between(future['ds'], future['yhat_lower'], future['yhat_upper'],
                color='#93C5FD', alpha=0.5, label='80% interval')

# Divider
split_date = daily['ds'].max()
ax.axvline(split_date, color='#9CA3AF', linewidth=1, linestyle='--')
ax.text(split_date + pd.Timedelta(days=0.5), ax.get_ylim()[1] * 0.92,
        'Forecast start', fontsize=9, color='#6B7280')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Daily inflow (₦)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₦{x/1000:.0f}k'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.set_title(f'Cash-Flow Forecast — {DEMO_ARCHETYPE.replace("_", " ").title()}\n'
             f'(Prophet · 180d history · 30d horizon · 80% CI)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(DATA_DIR / 'forecast_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → data/forecast_demo.png')

In [ ]:
# Prophet component decomposition — inspect trend + seasonalities
comp_fig = model.plot_components(forecast)
comp_fig.suptitle('Prophet Components — Trend, Weekly, Monthly, Holidays',
                  fontsize=11, fontweight='bold')
comp_fig.tight_layout()
plt.savefig(DATA_DIR / 'forecast_components.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/forecast_components.png')

## Step 5 — Backtest (500 Users)

Run this before trusting the model. If it's poorly calibrated you need to know before
the pitch, not after.

**Methodology:** single-window holdout — fit on all history except the last 30 days,
then score those 30 days against actuals. Needs ≥90 days of history to leave
enough training data for a meaningful fit.

**What to look for:**
- `MAE` — mean absolute error in naira
- `MASE` — MAE ÷ seasonal-naive MAE (repeat last 7 days). Below 1.0 means Prophet beats the naive baseline
- `Coverage` — fraction of actual values that land inside the 80% interval. Target: 0.75–0.85

**Red flags:**
| Symptom | Likely cause |
|---|---|
| MAE > 50% of mean actual | Model isn't learning; check data quality |
| MASE > 1.0 on most users | Prophet losing to naive; investigate the synthetic generator |
| Coverage < 0.70 | Intervals miscalibrated; check the log transform |

In [ ]:
from training.cash_flow_backtest import run_backtest, summarise

results_path = DATA_DIR / 'backtest_results.csv'

if results_path.exists():
    print('Loading pre-computed backtest results...')
    results = pd.read_csv(results_path)
else:
    # Select 500 users with enough history for backtest (≥90 + 30 = 120 days)
    eligible = (
        tx[tx['type'] == 'inflow']
        .groupby('user_id')['occurred_at']
        .count()
        .pipe(lambda s: s[s >= 120])
        .index.tolist()
    )
    sample = eligible[:500]
    print(f'Eligible users: {len(eligible):,}  →  testing {len(sample)}')
    print('Running backtest (~20 min for 500 users) ...')

    t0 = time.time()
    results = run_backtest(tx, sample)
    elapsed = time.time() - t0
    results.to_csv(results_path, index=False)
    print(f'Done in {elapsed/60:.1f} min. Saved → {results_path}')

print()
print('=== Backtest Summary ===')
summarise(results)

In [ ]:
# Distribution of key metrics across users
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.hist(results['mae_pct_of_mean'].dropna().clip(0, 1), bins=40, color='#2563EB', alpha=0.8)
ax.axvline(results['mae_pct_of_mean'].median(), color='#DC2626', linewidth=2,
           label=f'Median: {results["mae_pct_of_mean"].median():.1%}')
ax.set_xlabel('MAE as % of mean daily inflow')
ax.set_ylabel('Users')
ax.set_title('Forecast Error')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(results['mase'].dropna().clip(0, 2), bins=40, color='#059669', alpha=0.8)
ax.axvline(1.0, color='#9CA3AF', linewidth=1.5, linestyle='--', label='Naive baseline')
ax.axvline(results['mase'].median(), color='#DC2626', linewidth=2,
           label=f'Median: {results["mase"].median():.2f}')
ax.set_xlabel('MASE (< 1 = beats naive)')
ax.set_ylabel('Users')
ax.set_title('MASE vs Seasonal-Naive')
ax.legend(fontsize=9)

ax = axes[2]
ax.hist(results['coverage_80'].dropna(), bins=30, color='#7C3AED', alpha=0.8)
ax.axvspan(0.75, 0.85, color='#C4B5FD', alpha=0.3, label='Target 75–85%')
ax.axvline(results['coverage_80'].mean(), color='#DC2626', linewidth=2,
           label=f'Mean: {results["coverage_80"].mean():.1%}')
ax.set_xlabel('80% interval coverage')
ax.set_ylabel('Users')
ax.set_title('80% Coverage')
ax.legend(fontsize=9)

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Backtest Results — 500 Users, 30-Day Holdout', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_DIR / 'backtest_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/backtest_results.png')

In [ ]:
# Break down metrics by archetype to see which personas are hardest to forecast
results_with_arch = results.merge(
    users[['user_id', 'archetype']], on='user_id', how='left'
)

print('Metrics by archetype:')
display(
    results_with_arch.groupby('archetype')[['mae_pct_of_mean', 'mase', 'coverage_80']]
    .median()
    .round(3)
    .rename(columns={
        'mae_pct_of_mean': 'MAE% of mean',
        'mase': 'MASE',
        'coverage_80': 'Coverage@80%',
    })
    .style.background_gradient(cmap='RdYlGn_r', subset=['MAE% of mean', 'MASE'])
    .background_gradient(cmap='RdYlGn', subset=['Coverage@80%'])
)

## Step 6 — Dip Detection

A dip is a sustained income shortfall in the near-term forecast window.

**How it works:**
1. Take the user's **P25 daily inflow** over the last 60 days as the baseline
2. Flag each forecast day where `yhat_lower < 50% of baseline`
3. If **3 or more consecutive** flagged days appear within the next 14 days → dip detected
4. Severity scales with run length: `low` (3–4d), `medium` (5–6d), `high` (7d+)

`yhat_lower` is used rather than `yhat` by design — an offer only fires when
even the pessimistic end of the interval looks bad.
That keeps false-positive rates low and only surfaces real distress signals.

In [ ]:
from training.cash_flow import detect_dip

# Use the forecast we already computed for DEMO_USER
# Build history_df from the last 60 days of actual data
history_df = daily.tail(60)[['ds', 'y_orig']].rename(columns={'y_orig': 'y'})

# forecast already covers 30 future days
dip = detect_dip(forecast, history_df)

if dip:
    print('DIP DETECTED')
    for k, v in dip.items():
        print(f'  {k}: {v}')
else:
    print('No dip detected for demo user — try a user with more volatile cash flow below')

In [ ]:
# Synthetic dip test: manually construct a forecast with a clear dip
import pandas as pd, numpy as np
from training.cash_flow import detect_dip

today = pd.Timestamp.utcnow().normalize().tz_localize(None)
n = 14
synth_forecast = pd.DataFrame({
    'ds': pd.date_range(today, periods=n),
    'yhat': [10_000] * n,
    'yhat_lower': [8_000, 8_000, 2_000, 2_000, 1_500, 1_000, 2_000, 8_000] + [8_000] * 6,
    'yhat_upper': [14_000] * n,
})

# Baseline: P25 of last 60 days = ₦10,000 → threshold = ₦5,000
synth_history = pd.DataFrame({
    'ds': pd.date_range(today - pd.Timedelta(days=60), periods=60),
    'y':  np.random.normal(10_000, 2_000, 60).clip(0),
})

dip = detect_dip(synth_forecast, synth_history)
print('Synthetic dip result:')
for k, v in (dip or {}).items():
    print(f'  {k}: {v}')

In [ ]:
# Scan 50 random users — how many would trigger a dip warning today?
scan_users = users['user_id'].sample(50, random_state=42).tolist()
dip_users = []

hols = build_eom_holidays('2024-01-01', '2027-12-31')

for uid in scan_users:
    d = build_daily_series(tx, uid)
    if len(d) < 21:
        continue
    try:
        _, fc = fit_user_forecast(d, horizon_days=14, custom_holidays=hols)
        hist = d.tail(60)[['ds', 'y_orig']].rename(columns={'y_orig': 'y'})
        dip = detect_dip(fc, hist)
        if dip:
            dip_users.append({'user_id': uid, **dip})
    except Exception:
        pass

print(f'Scanned 50 users → {len(dip_users)} dip warnings')
if dip_users:
    display(pd.DataFrame(dip_users))

## Step 7 — Loan Sizing

Once a dip is detected, `size_loan` turns the predicted gap into a concrete offer amount.

**Formula:**
```
gap_naira  = sum(max(0, baseline − yhat) for each dip day)
raw_amount = gap_naira × 1.20
rounded    = round(raw_amount / 5,000) × 5,000
amount     = max(₦10,000, rounded)
amount     = min(amount, KudiScore cap)
```

The 20% buffer accounts for forecast uncertainty. Rounding to ₦5k keeps offers
feeling like round numbers to borrowers.

**KudiScore caps:**
| Score | Cap |
|---|---|
| ≥ 750 | ₦500k |
| ≥ 600 | ₦200k |
| ≥ 450 | ₦100k |
| < 450 | ₦50k |
| (no score) | ₦200k |

In [ ]:
from training.cash_flow import size_loan

# Demonstrate sizing at different gap amounts and KudiScores
test_cases = [
    (20_000, None, 'No score'),
    (20_000, 480,  'KudiScore 480'),
    (20_000, 620,  'KudiScore 620'),
    (20_000, 760,  'KudiScore 760'),
    (5_000,  None, 'Small gap (floor applies)'),
    (300_000, 800, 'Large gap (cap applies)'),
]

print(f'{"Gap (₦)":<15} {"KudiScore":<15} {"Loan (₦)":<15} Note')
print('-' * 60)
for gap, score, note in test_cases:
    loan_kobo = size_loan(gap, score)
    loan_naira = loan_kobo / 100
    print(f'₦{gap:<13,.0f} {str(score):<15} ₦{loan_naira:<13,.0f} {note}')

In [ ]:
# Show the full dip → loan pipeline for users flagged in the scan above
if dip_users:
    print('Dip warnings + loan offers:')
    print()
    for d in dip_users[:5]:
        gap_n = d['expected_gap_kobo'] / 100
        loan_n = d['suggested_loan_kobo'] / 100
        print(f"User: {d['user_id'][:8]}...")
        print(f"  Dip:  {d['dip_start_date']} → {d['dip_end_date']}  ({d['severity']})")
        print(f"  Gap:  ₦{gap_n:,.0f}")
        print(f"  Loan: ₦{loan_n:,.0f}")
        print()
else:
    print('No dip users from the scan. Try a larger sample or a period with more volatility.')

## Step 8 — End-to-End: CashFlowPredictor

`POST /predict/forecast` in `app.py` calls `CashFlowPredictor.predict()`, which runs
the full serving path in order:

1. **Cache lookup** — return the cached forecast if `fit_at` is within TTL
2. **Lazy fit** — fit Prophet on demand if there's no cache hit
3. **Cold-start** — fall back to the archetype profile if history < 21 days
4. **Dip detection** — scan the forecast window for sustained shortfalls
5. **Cache write** — store the result so the next request is instant

Run this step with a mock DB backed by an in-memory dict so you can see the
full flow without a live Postgres instance.

In [ ]:
# Minimal mock DB for offline demo (replaces SQLAlchemy Session)
from datetime import datetime, timedelta

class MockDB:
    """In-memory stand-in for SQLAlchemy Session. Stores forecast cache in a dict."""
    def __init__(self, tx_df, users_df):
        self.tx_df    = tx_df
        self.users_df = users_df
        self._cache   = {}   # {user_id: [row_dicts]}

# Monkey-patch the db functions to use our mock
import db as db_module

_MOCK_CACHE = {}

def _mock_fetch_transactions(db, user_id, as_of):
    user_tx = db.tx_df[db.tx_df['user_id'] == user_id].copy()
    user_tx = user_tx[user_tx['occurred_at'] < as_of]
    return user_tx.rename(columns={'occurred_at': 'occurred_at'})

def _mock_fetch_user_meta(db, user_id):
    row = db.users_df[db.users_df['user_id'] == user_id]
    if row.empty: return {}
    r = row.iloc[0]
    return {'archetype': r['archetype'], 'market_location': r.get('market_location', ''),
            'gender': r.get('gender', ''), 'age_bracket': r.get('age_bracket', ''),
            'onboarding_date': r.get('onboarding_date', None)}

def _mock_write_forecast_cache(db, user_id, forecast_df, model_version):
    now = datetime.utcnow()
    _MOCK_CACHE[user_id] = [
        {
            'forecast_date': row['ds'].date(),
            'predicted_inflow': int(round(row['yhat'] * 100)),
            'lower_bound_80': int(round(row['yhat_lower'] * 100)),
            'upper_bound_80': int(round(row['yhat_upper'] * 100)),
            'model_version': model_version,
            '_fit_at': now,
        }
        for _, row in forecast_df.iterrows()
    ]

def _mock_read_forecast_cache(db, user_id, fresh_within_hours=24):
    rows = _MOCK_CACHE.get(user_id, [])
    cutoff = datetime.utcnow() - timedelta(hours=fresh_within_hours)
    return [r for r in rows if r.get('_fit_at', datetime.min) >= cutoff]

def _mock_fetch_user_daily_history(db, user_id, days=60):
    cutoff = datetime.utcnow() - timedelta(days=days)
    user_tx = db.tx_df[
        (db.tx_df['user_id'] == user_id)
        & (db.tx_df['type'] == 'inflow')
        & (db.tx_df['occurred_at'] >= cutoff)
    ].copy()
    if user_tx.empty:
        return pd.DataFrame(columns=['ds', 'y'])
    user_tx['date'] = user_tx['occurred_at'].dt.normalize()
    daily = user_tx.groupby('date')['amount_kobo'].sum().rename('y') / 100.0
    out = daily.reset_index()
    out.columns = ['ds', 'y']
    return out

# Patch db module
db_module.fetch_transactions       = _mock_fetch_transactions
db_module.fetch_user_meta          = _mock_fetch_user_meta
db_module.write_forecast_cache     = _mock_write_forecast_cache
db_module.read_forecast_cache      = _mock_read_forecast_cache
db_module.fetch_user_daily_history = _mock_fetch_user_daily_history

print('Mock DB ready.')

In [ ]:
from inference.cash_flow_predictor import CashFlowPredictor

predictor = CashFlowPredictor.load(profiles_path=DATA_DIR / 'archetype_profiles.json')
mock_db   = MockDB(tx, users)

# ── First call: cold cache → lazy fit ────────────────────────────────────────
print('=== First call (cold cache — lazy fit) ===')
t0 = time.time()
result = predictor.predict(DEMO_USER, mock_db, horizon_days=30)
print(f'Time: {time.time()-t0:.1f}s')
print(f'model_version: {result["model_version"]}')
print(f'daily rows:    {len(result["daily"])}')
print(f'dip_warning:   {result["dip_warning"]}')
print()

# ── Second call: cache hit ────────────────────────────────────────────────────
print('=== Second call (cache hit — no refit) ===')
t0 = time.time()
result2 = predictor.predict(DEMO_USER, mock_db, horizon_days=30)
print(f'Time: {time.time()-t0:.2f}s')
print(f'model_version: {result2["model_version"]}')
print(f'Same daily rows: {result["daily"] == result2["daily"]}')

In [ ]:
# Demo: cold-start user (find a user with <21 days of history, or create a synthetic one)
_MOCK_CACHE.clear()   # reset cache

# Build a tiny synthetic TX dataset for a brand-new user
NEW_USER_ID = 'new-user-demo-0001'
new_user_tx = pd.DataFrame({
    'user_id': [NEW_USER_ID] * 5,
    'occurred_at': pd.date_range('2026-05-08', periods=5),
    'amount_kobo': [250_000, 180_000, 310_000, 200_000, 270_000],
    'type': 'inflow',
    'sender_name': 'customer',
    'is_repeat_customer': False,
})
new_user_row = pd.DataFrame([{
    'user_id': NEW_USER_ID,
    'archetype': 'market_food_vendor',
    'onboarding_date': pd.Timestamp('2026-05-08'),
    'gender': 'female', 'age_bracket': 2, 'market_location': 'Lagos Island'
}])

cold_db = MockDB(
    pd.concat([tx, new_user_tx], ignore_index=True),
    pd.concat([users, new_user_row], ignore_index=True),
)

print(f'Cold-start user has {len(new_user_tx)} days of history (< 21 → archetype fallback)')
result_cold = predictor.predict(NEW_USER_ID, cold_db, horizon_days=14)
print(f'model_version: {result_cold["model_version"]}')
print(f'daily rows:    {len(result_cold["daily"])}')
print()

print('First 7 forecast days (archetype-based):')
for r in result_cold['daily'][:7]:
    naira = r['predicted_inflow'] / 100
    lo    = r['lower_bound_80'] / 100
    hi    = r['upper_bound_80'] / 100
    print(f"  {r['forecast_date']}  yhat=₦{naira:>8,.0f}  [{lo:>8,.0f} – {hi:>8,.0f}]")

## Summary

### Components

| File | What it does |
|---|---|
| `training/cash_flow.py` | Daily series, log1p transform, Prophet fit, archetypes, dip detection, loan sizing |
| `training/cash_flow_batch.py` | Nightly joblib-parallel mass fit |
| `training/cash_flow_backtest.py` | 30-day holdout evaluation |
| `inference/cash_flow_predictor.py` | `CashFlowPredictor` — cache-first serving class |
| `app.py` | `POST /predict/forecast` route |
| `db.py` | Cache read/write, daily history fetch |
| `migrations/002_forecasting_tables.sql` | `forecasts` + `dip_warnings` tables |

### Before going live
```bash
# 1. Run the DB migration
psql $DATABASE_URL -f migrations/002_forecasting_tables.sql

# 2. Run the nightly batch (or schedule it for 2 AM Lagos time)
python -m training.cash_flow_batch

# 3. Re-run the backtest with more users
python -m training.cash_flow_backtest
```

---

### Troubleshooting

| Symptom | Likely cause |
|---|---|
| `yhat_lower` is negative | Missing log transform or the `clip(lower=0)` |
| Coverage at 80% is ~0.55 | No log transform on right-skewed data |
| Predictions in millions/day | Forgot to divide kobo by 100 before fit |
| Flat horizontal predictions | Fitting on ≤7 days; check the `n_days < 21` guard |
| Fit takes 5+ seconds per user | Yearly seasonality is on; turn it off |
| All users get the same archetype forecast | `persona_map` key type mismatch (str vs int) |
| `cmdstanpy` errors on import | Stan didn't compile; reinstall prophet |
| MASE > 1.0 on most users | Prophet losing to naive; check the synthetic generator |